In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.functions import sum as _sum, count, col
from pyspark.sql.window import Window
from pyspark.sql.types import *

### Step 3 - Define the schema

In [0]:
transaction_schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("event_time",     TimestampType(), False),
    StructField("category",       StringType(), False),
    StructField("amount",         DoubleType(), False),
    StructField("quantity",       IntegerType(), False)
])

### Step 4 - ReadStream using Auto Loader

👉 **.schema(transaction_schema)**
Used for:
- Initial schema definition
- Data type enforcement
- Avoid bad data inference
- Production stability

👉 **cloudFiles.schemaLocation**
Used for:
- Storing inferred schema state
- Handling schema evolution
- Remembering schema across runs

**_Auto Loader (cloudFiles)_** is Databricks' optimized file streaming source — it tracks which files have already been processed so you never double-count, giving you exactly-once ingestion.

In [0]:
raw_stream = (
    spark.readStream
        .format("cloudFiles")                   
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation",
                "/Volumes/streaming_demo/sales/checkpoints/schema")
        .schema(transaction_schema)
        .load("/Volumes/streaming_demo/sales/raw_transactions/")
)

### Step 5 — Transform with Windowed Aggregation

In [0]:


aggregated = (
    raw_stream
        .withWatermark("event_time", "2 minutes")   # tolerate 2-min late data
        .groupBy(
            window(col("event_time"), "1 minute"),  # tumbling 1-min window
            col("category")
        )
        .agg(
            _sum("amount").alias("total_revenue"),
            count("transaction_id").alias("order_count")
        )
        .select(
            col("window.start").alias("window_start"),
            col("window.end").alias("window_end"),
            col("category"),
            col("total_revenue"),
            col("order_count")
        )
)

### Step 6 — Write Stream to Delta Table

**NOTE: Limitiations in Free Edition to implement writeStream**
1. In serverless edition Databricks doesn't allow us to run by default trigger options hence we need to explicitly specify trigger option **(.trigger(availableNow=True))** during writeStream.
2. It also doesn't allow us to use .outputMode("update") instead we need to use **.outputMode("append")**

In [0]:
checkpoint_path = "/Volumes/streaming_demo/sales/checkpoints/agg_query"

query = (
    aggregated.writeStream
        .format("delta")
        .trigger(availableNow=True)
        .outputMode("append")      
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .toTable("streaming_demo.sales.revenue_by_category") 
)

print("✅ Streaming query started:", query.id)

### Step 7 — Observe & Query Results

In [0]:
%sql
SELECT category, round(total_revenue,2) as total_revenue, order_count
FROM streaming_demo.sales.revenue_by_category
order by total_revenue desc;